# Customer Churn Prediction System

### Data Cleaning, Train/Test Split & Feature Engineering

## Objective

The objective of this notebook is to perform data cleaning based on data quality audit findings, split the dataset into training and testing sets to prevent data leakage, and construct new features to improve model performance.

At this stage, no feature encoding, scaling, or machine learning modeling will be applied. The focus is strictly on preparing clean datasets and creating domain features before moving to preprocessing and baseline modeling.

In [24]:
import pandas as pd
import numpy as np

print(pd.__version__)
print(np.__version__)

3.0.5
2.5.1


In [25]:
from pathlib import Path
Path.cwd()

WindowsPath('c:/AI-Projects/customer-churn-prediction-system/notebooks')

In [26]:
data_path = Path("../data/WA_Fn-UseC_-Telco-Customer-Churn.csv")

print(data_path)

data_path.exists()

..\data\WA_Fn-UseC_-Telco-Customer-Churn.csv


True

In [27]:
df = pd.read_csv(data_path)

In [ ]:
numerical_features = df.select_dtypes(include="number").columns
categorical_features = df.select_dtypes(include = 'object')

In [29]:
print("Shape:", df.shape)

df.info()

display(df.head())

Shape: (7043, 21)
<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   s

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [30]:
missing_values = df.isnull().sum()

missing_percentage = (missing_values / len(df)) * 100

missing_summary = pd.DataFrame({
    "missing_count": missing_values,
    "missing_percentage": missing_percentage
})

missing_summary[missing_summary["missing_count"] >= 0]

,missing_count,missing_percentage
customerID,0,0.0
gender,0,0.0
SeniorCitizen,0,0.0
Partner,0,0.0
Dependents,0,0.0
tenure,0,0.0
PhoneService,0,0.0
MultipleLines,0,0.0
InternetService,0,0.0
OnlineSecurity,0,0.0


In [31]:
duplicate_count = df.duplicated().sum()

print("Duplicate rows:", duplicate_count)

Duplicate rows: 0


In [32]:
for column in categorical_features:
    print(f"\n--- {column} ---")
    print("Unique values:", df[column].nunique())
    print(df[column].unique())


--- customerID ---
Unique values: 7043
<StringArray>
['7590-VHVEG', '5575-GNVDE', '3668-QPYBK', '7795-CFOCW', '9237-HQITU',
 '9305-CDSKC', '1452-KIOVK', '6713-OKOMC', '7892-POOKP', '6388-TABGU',
 ...
 '9767-FFLEM', '0639-TSIQW', '8456-QDAVC', '7750-EYXWZ', '2569-WGERO',
 '6840-RESVB', '2234-XADUH', '4801-JZAZL', '8361-LTMKD', '3186-AJIEK']
Length: 7043, dtype: str

--- gender ---
Unique values: 2
<StringArray>
['Female', 'Male']
Length: 2, dtype: str

--- Partner ---
Unique values: 2
<StringArray>
['Yes', 'No']
Length: 2, dtype: str

--- Dependents ---
Unique values: 2
<StringArray>
['No', 'Yes']
Length: 2, dtype: str

--- PhoneService ---
Unique values: 2
<StringArray>
['No', 'Yes']
Length: 2, dtype: str

--- MultipleLines ---
Unique values: 3
<StringArray>
['No phone service', 'No', 'Yes']
Length: 3, dtype: str

--- InternetService ---
Unique values: 3
<StringArray>
['DSL', 'Fiber optic', 'No']
Length: 3, dtype: str

--- OnlineSecurity ---
Unique values: 3
<StringArray>
['No', 'Yes'

In [33]:
print("Target unique values:")
print(df["Churn"].unique())

print("\nTarget distribution:")
print(df["Churn"].value_counts(dropna=False))

Target unique values:
<StringArray>
['No', 'Yes']
Length: 2, dtype: str

Target distribution:
Churn
No     5174
Yes    1869
Name: count, dtype: int64


In [34]:
display(df[numerical_features].describe())

,SeniorCitizen,tenure,MonthlyCharges
count,7043.000000,7043.000000,7043.000000
mean,0.162147,32.371149,64.761692
std,0.368612,24.559481,30.090047
min,0.000000,0.000000,18.250000
25%,0.000000,9.000000,35.500000
50%,0.000000,29.000000,70.350000
75%,0.000000,55.000000,89.850000
max,1.000000,72.000000,118.750000


In [35]:
print("TotalCharges dtype:", df["TotalCharges"].dtype)

converted = pd.to_numeric(df["TotalCharges"], errors="coerce")

invalid_mask = converted.isna()

print("Non-numeric / invalid values:", invalid_mask.sum())

TotalCharges dtype: str
Non-numeric / invalid values: 11


In [36]:
print("\nInvalid values:")

print(df.loc[invalid_mask, "TotalCharges"].unique())


Invalid values:
<StringArray>
[' ']
Length: 1, dtype: str


In [37]:
display(
    df.loc[
        invalid_mask
    ]
)

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
488,4472-LVYGI,Female,0,Yes,Yes,0,No,No phone service,DSL,Yes,...,Yes,Yes,Yes,No,Two year,Yes,Bank transfer (automatic),52.55,,No
753,3115-CZMZD,Male,0,No,Yes,0,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,20.25,,No
936,5709-LVOEQ,Female,0,Yes,Yes,0,Yes,No,DSL,Yes,...,Yes,No,Yes,Yes,Two year,No,Mailed check,80.85,,No
1082,4367-NUYAO,Male,0,Yes,Yes,0,Yes,Yes,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,25.75,,No
1340,1371-DWPAZ,Female,0,Yes,Yes,0,No,No phone service,DSL,Yes,...,Yes,Yes,Yes,No,Two year,No,Credit card (automatic),56.05,,No
3331,7644-OMVMY,Male,0,Yes,Yes,0,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,19.85,,No
3826,3213-VVOLG,Male,0,Yes,Yes,0,Yes,Yes,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,25.35,,No
4380,2520-SGTTA,Female,0,Yes,Yes,0,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,20.00,,No
5218,2923-ARZLG,Male,0,Yes,Yes,0,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,One year,Yes,Mailed check,19.70,,No
6670,4075-WKNIU,Female,0,Yes,Yes,0,Yes,Yes,DSL,No,...,Yes,Yes,Yes,No,Two year,No,Mailed check,73.35,,No


In [38]:
display(
    df.loc[
        invalid_mask,
        ["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Churn"]
    ]
)

,customerID,tenure,MonthlyCharges,TotalCharges,Churn
488,4472-LVYGI,0,52.55,,No
753,3115-CZMZD,0,20.25,,No
936,5709-LVOEQ,0,80.85,,No
1082,4367-NUYAO,0,25.75,,No
1340,1371-DWPAZ,0,56.05,,No
3331,7644-OMVMY,0,19.85,,No
3826,3213-VVOLG,0,25.35,,No
4380,2520-SGTTA,0,20.00,,No
5218,2923-ARZLG,0,19.70,,No
6670,4075-WKNIU,0,73.35,,No


In [39]:
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
).fillna(0)

In [40]:
print("TotalCharges dtype:", df["TotalCharges"].dtype)
print("Missing values:", df["TotalCharges"].isna().sum())
print("Zero TotalCharges:", (df["TotalCharges"] == 0).sum())

TotalCharges dtype: float64
Missing values: 0
Zero TotalCharges: 11


In [41]:
print("Shape:", df.shape)

print("\nMissing values:")
print(df.isnull().sum()[df.isnull().sum() > 0])

print("\nDuplicate rows:", df.duplicated().sum())

print("\nData types:")
print(df.dtypes)

print("\nTotalCharges dtype:", df["TotalCharges"].dtype)

Shape: (7043, 21)

Missing values:
Series([], dtype: int64)

Duplicate rows: 0

Data types:
customerID              str
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges        float64
Churn                   str
dtype: object

TotalCharges dtype: float64


In [42]:
from sklearn.model_selection import train_test_split

# 1. Separate Features (X) and Target (y)
X = df.drop(columns=["Churn"])
y = df["Churn"]

# 2. Perform Stratified Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# 3. Validation
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("\nTrain Churn distribution (%):\n", y_train.value_counts(normalize=True) * 100)
print("\nTest Churn distribution (%):\n", y_test.value_counts(normalize=True) * 100)

X_train shape: (5634, 20)
X_test shape: (1409, 20)

Train Churn distribution (%):
 Churn
No     73.464679
Yes    26.535321
Name: proportion, dtype: float64

Test Churn distribution (%):
 Churn
No     73.456352
Yes    26.543648
Name: proportion, dtype: float64


In [43]:

#chek for index overlab between train and test
index_overlab = len(set(X_train.index).intersection(set(X_test.index)))
print("index_overlab_count :", index_overlab)

#check customerID in train features
print("\nIs customerID in features?" ,"customerID" in X_train.columns)

#check numerical and categorecal features in training set
print("\nX_train info summary : ")
print(X_train.info())

index_overlab_count : 0

Is customerID in features? True

X_train info summary : 
<class 'pandas.DataFrame'>
Index: 5634 entries, 3738 to 5639
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        5634 non-null   str    
 1   gender            5634 non-null   str    
 2   SeniorCitizen     5634 non-null   int64  
 3   Partner           5634 non-null   str    
 4   Dependents        5634 non-null   str    
 5   tenure            5634 non-null   int64  
 6   PhoneService      5634 non-null   str    
 7   MultipleLines     5634 non-null   str    
 8   InternetService   5634 non-null   str    
 9   OnlineSecurity    5634 non-null   str    
 10  OnlineBackup      5634 non-null   str    
 11  DeviceProtection  5634 non-null   str    
 12  TechSupport       5634 non-null   str    
 13  StreamingTV       5634 non-null   str    
 14  StreamingMovies   5634 non-null   str    
 15  Contract          56

In [44]:
import os

# 1. Create directory
os.makedirs("../data/processed", exist_ok=True)

# 2. Save Split sets to CSV
train_df = pd.concat([X_train, y_train], axis=1)
test_df = pd.concat([X_test, y_test], axis=1)

train_df.to_csv("../data/processed/train.csv", index=False)
test_df.to_csv("../data/processed/test.csv", index=False)

print("Train and Test sets successfully saved to 'data/processed/'!")

Train and Test sets successfully saved to 'data/processed/'!


In [45]:
import pandas as pd
import numpy as np

# 1. Define Feature Engineering Function
def create_engineered_features(df_input):
    df_out = df_input.copy()
    
    # Feature 1: Total Services Count
    service_cols = [
        'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 
        'TechSupport', 'StreamingTV', 'StreamingMovies'
    ]
    df_out['TotalServices'] = (df_out[service_cols] == 'Yes').sum(axis=1)
    
    # Feature 2: Automatic Payment Indicator
    auto_payments = ['Bank transfer (automatic)', 'Credit card (automatic)']
    df_out['IsAutomaticPayment'] = df_out['PaymentMethod'].isin(auto_payments).astype(int)
    
    # Feature 3: Monthly Spend Difference (Current vs Historical Avg)
    # Using tenure + 1 to safely prevent division by zero for tenure = 0
    historical_avg = df_out['TotalCharges'] / (df_out['tenure'] + 1)
    df_out['MonthlySpendDiff'] = df_out['MonthlyCharges'] - historical_avg
    
    return df_out

# 2. Apply to X_train and X_test independently
X_train_fe = create_engineered_features(X_train)
X_test_fe = create_engineered_features(X_test)

# 3. Validation
print("Original X_train shape:", X_train.shape)
print("Engineered X_train shape:", X_train_fe.shape)
print("\nSample of engineered features:")
display(X_train_fe[['TotalServices', 'IsAutomaticPayment', 'MonthlySpendDiff']].head())

Original X_train shape: (5634, 20)
Engineered X_train shape: (5634, 23)

Sample of engineered features:


,TotalServices,IsAutomaticPayment,MonthlySpendDiff
3738,3,0,1.931944
3151,1,0,3.128125
4860,3,0,-1.617857
3867,4,1,2.918519
3810,0,0,22.275000


In [46]:
# 1. Check for Missing (NaN) or Infinite (Inf) values in new features
new_cols = ['TotalServices', 'IsAutomaticPayment', 'MonthlySpendDiff']

print("--- Train Set Check ---")
print("Missing values:", X_train_fe[new_cols].isna().sum().to_dict())
print("Infinite values:", np.isinf(X_train_fe[new_cols]).sum().to_dict())

print("\n--- Test Set Check ---")
print("Missing values:", X_test_fe[new_cols].isna().sum().to_dict())
print("Infinite values:", np.isinf(X_test_fe[new_cols]).sum().to_dict())

# 2. Summary Statistics for Engineered Features
print("\n--- Train Summary Stats ---")
display(X_train_fe[new_cols].describe().T[['min', 'mean', 'max']])

--- Train Set Check ---
Missing values: {'TotalServices': 0, 'IsAutomaticPayment': 0, 'MonthlySpendDiff': 0}
Infinite values: {'TotalServices': 0, 'IsAutomaticPayment': 0, 'MonthlySpendDiff': 0}

--- Test Set Check ---
Missing values: {'TotalServices': 0, 'IsAutomaticPayment': 0, 'MonthlySpendDiff': 0}
Infinite values: {'TotalServices': 0, 'IsAutomaticPayment': 0, 'MonthlySpendDiff': 0}

--- Train Summary Stats ---


,min,mean,max
TotalServices,0.000000,2.058040,6.00
IsAutomaticPayment,0.000000,0.436102,1.00
MonthlySpendDiff,-6.456667,5.723199,73.35


In [47]:
import os

# 1. Combine Features and Target for Export
train_fe_df = pd.concat([X_train_fe, y_train], axis=1)
test_fe_df = pd.concat([X_test_fe, y_test], axis=1)

# 2. Save Updated Datasets
train_fe_df.to_csv("../data/processed/train_engineered.csv", index=False)
test_fe_df.to_csv("../data/processed/test_engineered.csv", index=False)

print("Engineered Train & Test datasets saved successfully to 'data/processed/'!")
print("Final Train shape:", train_fe_df.shape)
print("Final Test shape:", test_fe_df.shape)

Engineered Train & Test datasets saved successfully to 'data/processed/'!
Final Train shape: (5634, 24)
Final Test shape: (1409, 24)
